# 🔎 Vector Search Demo — FAISS (Local) + Pinecone (Cloud)

This notebook demonstrates two approaches to vector similarity search:

1. **FAISS** — a local, in-memory vector search library (no internet/API key needed)
2. **Pinecone** — a managed cloud vector database (requires a free API key)

We'll embed a small set of text documents into vectors using a Sentence-Transformers model, then run similarity search using different index types and filtering strategies.

**How to use this notebook:**
- Run cells top to bottom
- Section 1 installs dependencies
- Section 2 (FAISS) works immediately, no setup required
- Section 3 (Pinecone) requires you to paste in your own API key


---
## Section 1 — Install Dependencies

Run this once per Colab session. This installs:
- `faiss-cpu` — local vector index library
- `sentence-transformers` — for generating text embeddings
- `pinecone` — Python client for the Pinecone cloud vector database


In [1]:
# Install required packages (only needs to run once per Colab session)
!pip install faiss-cpu sentence-transformers pinecone -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 11.6 MB/s eta 0:00:00


### Imports

In [2]:
import numpy as np                                  # numerical arrays for embeddings
import faiss                                         # local vector search library
import time                                          # used to timestamp metadata
from sentence_transformers import SentenceTransformer  # turns text into embeddings
from pinecone import Pinecone, ServerlessSpec        # Pinecone cloud vector DB client

---
## Section 2 — FAISS (Local Vector Search)

FAISS (Facebook AI Similarity Search) lets you build and query vector indexes **entirely on your own machine** — no network calls, no API keys.

We'll compare three index types on the same tiny dataset:

| Index Type | How it works | Best for |
|---|---|---|
| **Flat (IndexFlatL2)** | Brute-force exact search — compares the query to every vector | Small datasets, perfect accuracy |
| **HNSW** | Graph-based approximate search | Large datasets, fast + accurate |
| **IVF** | Clusters vectors, searches only nearby clusters | Very large datasets, tunable speed/accuracy tradeoff |


### 2.1 Prepare the dataset and embeddings

In [3]:
print("\n========== FAISS DEMO ==========\n")

# A small toy dataset of documents to search over
texts = [
    "Document about AI simulations.",
    "Deep learning and neural networks.",
    "Vector databases and similarity search.",
    "Graph-based indexing methods.",
    "Clustering algorithms in machine learning."
]

# Load a lightweight pretrained embedding model
# "all-MiniLM-L6-v2" maps sentences to 384-dimensional dense vectors
model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert each text document into a vector embedding
embeddings = model.encode(texts)
embeddings = np.array(embeddings).astype("float32")  # FAISS requires float32

# All FAISS indexes need to know the vector dimensionality up front
dimension = embeddings.shape[1]
print(f"Embedding dimension: {dimension}")


========== FAISS DEMO ==========



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


### 2.2 Flat Index — Exact Search

`IndexFlatL2` computes the exact Euclidean (L2) distance between the query and *every* stored vector.
It's slow at scale but guarantees the most accurate results — good as a ground-truth baseline.


In [4]:
print("---- FAISS: Flat Index ----")

# Create a flat (brute-force) index using L2 distance
flat_index = faiss.IndexFlatL2(dimension)
flat_index.add(embeddings)  # add our document vectors to the index

# Embed the search query the same way we embedded the documents
query = "AI systems"
query_vector = model.encode([query]).astype("float32")

k = 3  # number of nearest neighbors to retrieve

# Search: returns distances and indices of the k closest vectors
distances, indices = flat_index.search(query_vector, k)

# Print the matching document texts
for i in indices[0]:
    print(texts[i])

---- FAISS: Flat Index ----
Document about AI simulations.
Deep learning and neural networks.
Clustering algorithms in machine learning.


### 2.3 HNSW Index — Graph-Based Approximate Search

`IndexHNSWFlat` builds a navigable small-world graph over the vectors.
It's much faster than Flat search on large datasets while staying highly accurate — a common production choice.

The `32` parameter below controls the number of graph connections per node (higher = more accurate but more memory).


In [5]:
print("\n---- FAISS: HNSW Index ----")

# Build an HNSW graph index; 32 = number of neighbor links per node
hnsw_index = faiss.IndexHNSWFlat(dimension, 32)
hnsw_index.add(embeddings)

# Reuse the same query vector from before
distances, indices = hnsw_index.search(query_vector, k)

for i in indices[0]:
    print(texts[i])


---- FAISS: HNSW Index ----
Document about AI simulations.
Deep learning and neural networks.
Clustering algorithms in machine learning.


### 2.4 IVF Index — Cluster-Based Search

`IndexIVFFlat` first clusters all vectors into `nlist` groups (via k-means), then at search time only
looks inside the closest clusters (`nprobe` controls how many). This trades a little accuracy for a lot of speed
on large datasets.

⚠️ Note: With only 5 documents and 2 clusters, this is just for demonstration — IVF really shines with much larger datasets.


In [6]:
print("\n---- FAISS: IVF Index ----")

nlist = 2  # number of clusters to partition the vectors into

# IVF needs a "quantizer" index to assign vectors to clusters
quantizer = faiss.IndexFlatL2(dimension)
ivf_index = faiss.IndexIVFFlat(quantizer, dimension, nlist)

# IVF indexes must be trained (to learn the cluster centroids) before adding data
ivf_index.train(embeddings)
ivf_index.add(embeddings)

ivf_index.nprobe = 1  # how many clusters to search at query time (higher = more accurate, slower)

distances, indices = ivf_index.search(query_vector, k)

for i in indices[0]:
    print(texts[i])


---- FAISS: IVF Index ----
Document about AI simulations.
Deep learning and neural networks.
Clustering algorithms in machine learning.


---
## Section 3 — Pinecone (Cloud Vector Database)

Unlike FAISS, Pinecone is a **managed cloud service** — your vectors are stored remotely and queried over the network.
This means you get persistence, scalability, and metadata filtering without managing infrastructure yourself.

**You'll need a free Pinecone API key:** sign up at [pinecone.io](https://www.pinecone.io/) and paste your key below.

> 🔐 **Security tip:** Don't hardcode API keys in a notebook you plan to share. On Colab, use `google.colab.userdata` (Secrets) or an environment variable instead of pasting it directly into code.


### 3.1 Connect to Pinecone and create an index

In [7]:
from google.colab import userdata
print("\n========== PINECONE DEMO ==========\n")

# 🔐 Replace with your own Pinecone API key
api_key = "YOUR_API_KEY"

# Initialize the Pinecone client
pc = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))

index_name = "demo-index"

# Only create the index if it doesn't already exist (avoids errors on re-run)
if index_name not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,              # must match our embedding model's output size
        metric="cosine",            # similarity metric: cosine, euclidean, or dotproduct
        spec=ServerlessSpec(cloud="aws", region="us-east-1")  # serverless = pay-per-use, no infra to manage
    )

# Get a handle to the index so we can upsert/query it
index = pc.Index(index_name)


========== PINECONE DEMO ==========



### 3.2 Upload (upsert) documents with metadata

Each vector in Pinecone can carry arbitrary **metadata** (a dict of key/value pairs) alongside it.
This lets us filter search results later — e.g. "only show me AI-related documents".


In [9]:
# Sample dataset to upload to Pinecone
pinecone_texts = [
    "AI is transforming healthcare",
    "Robotics in industrial automation",
    "Machine learning in finance",
    "AI for simulation and modeling",
    "Cloud computing basics"
]

# Embed each document and upload it with metadata attached
for i, text in enumerate(pinecone_texts):
    embedding = model.encode(text).tolist()  # Pinecone expects a plain list, not a numpy array

    # Metadata used later for filtering search results
    metadata = {
        "category": "ai" if "AI" in text else "other",  # simple category tag
        "created_at": int(time.time()),                 # unix timestamp of insertion
        "keywords": text.lower().split()                 # list of lowercase words for keyword filtering
    }

    # upsert = "insert or update" — each vector needs a unique ID
    index.upsert(vectors=[
    (f"doc-{i}", embedding, metadata)
    ])

print(f"Upserted {len(pinecone_texts)} documents into '{index_name}'")

Upserted 5 documents into 'demo-index'


### 3.3 Basic Semantic Search

A plain nearest-neighbor query with no filters — returns the `top_k` most semantically similar documents to the query.


In [12]:
print("---- Pinecone: Semantic Search ----")

query = "AI in simulations"
query_embedding = model.encode(query).tolist()

results = index.query(
    vector=query_embedding,
    top_k=3,               # return the 3 closest matches
    include_metadata=True  # also return each match's metadata dict
)

for match in results["matches"]:
    print("Id = ", match["id"] ," with score : ", match["score"])  # id = doc identifier, score = similarity score
    print("Metadata = " , match["metadata"])
    print("-----")

---- Pinecone: Semantic Search ----
Id =  doc-3  with score :  0.875126302
Metadata =  {'category': 'ai', 'created_at': 1789620361, 'keywords': ['ai', 'for', 'simulation', 'and', 'modeling']}
-----
Id =  doc-0  with score :  0.394708812
Metadata =  {'category': 'ai', 'created_at': 1789620360, 'keywords': ['ai', 'is', 'transforming', 'healthcare']}
-----
Id =  doc-1  with score :  0.324862391
Metadata =  {'category': 'other', 'created_at': 1789620361, 'keywords': ['robotics', 'in', 'industrial', 'automation']}
-----


### 3.4 Filter by Category

Metadata filters narrow results to only vectors whose metadata matches a condition — here, `category == "ai"`.
This runs *alongside* the similarity search, so you still get semantically relevant results, just restricted to a subset.


In [14]:
print("\n---- Pinecone: Filter by Category ----")

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
    filter={
        "category": {"$eq": "ai"}   # only return docs where category equals "ai"
    }
)

for match in results["matches"]:
    print("Id = ", match["id"] ," with score : ", match["score"])  # id = doc identifier, score = similarity score
    print("Metadata = " , match["metadata"])
    print("-----")


---- Pinecone: Filter by Category ----
Id =  doc-3  with score :  0.875126302
Metadata =  {'category': 'ai', 'created_at': 1789620361, 'keywords': ['ai', 'for', 'simulation', 'and', 'modeling']}
-----
Id =  doc-0  with score :  0.394708812
Metadata =  {'category': 'ai', 'created_at': 1789620360, 'keywords': ['ai', 'is', 'transforming', 'healthcare']}
-----


### 3.5 Filter by Keyword

The `$in` operator checks whether the `keywords` metadata list contains any of the given values —
here, only documents whose keyword list includes "finance" are returned.


In [15]:
print("\n---- Pinecone: Filter by Keyword ----")

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
    filter={
        "keywords": {"$in": ["finance"]}   # only return docs that mention "finance"
    }
)

for match in results["matches"]:
    print("Id = ", match["id"] ," with score : ", match["score"])  # id = doc identifier, score = similarity score
    print("Metadata = " , match["metadata"])
    print("-----")


---- Pinecone: Filter by Keyword ----
Id =  doc-2  with score :  0.268622309
Metadata =  {'category': 'other', 'created_at': 1789620361, 'keywords': ['machine', 'learning', 'in', 'finance']}
-----


### 3.6 Combined Filter (AND logic)

Multiple filter conditions can be combined with `$and` to require that **all** conditions match —
here: category is "ai" **and** the keyword list includes "simulation".


In [16]:
print("\n---- Pinecone: Combined Filter ----")

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
    filter={
        "$and": [
            {"category": {"$eq": "ai"}},          # condition 1: category must be "ai"
            {"keywords": {"$in": ["simulation"]}} # condition 2: keywords must include "simulation"
        ]
    }
)

for match in results["matches"]:
    print("Id = ", match["id"] ," with score : ", match["score"])  # id = doc identifier, score = similarity score
    print("Metadata = " , match["metadata"])
    print("-----")


---- Pinecone: Combined Filter ----
Id =  doc-3  with score :  0.875126302
Metadata =  {'category': 'ai', 'created_at': 1789620361, 'keywords': ['ai', 'for', 'simulation', 'and', 'modeling']}
-----


---
## Done ✅

In [17]:
print("\n========== DONE ==========")


========== DONE ==========
